In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import random
import cv2

from src.data import recordid_to_imageids, imgid_to_recid
from src.visualization import imgid_to_imgarray
from src import DATA_BASE_DIR as data_base_dir
from src import logger
import matplotlib.pyplot as plt
from matplotlib import gridspec

# Add project root to sys.path (so imports like src.data work)
project_root = Path().cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Font

In [ ]:
from matplotlib import font_manager as fm
fm.fontManager.addfont("plots/times.ttf")

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": "Times New Roman",
    "font.size": 10,             # main text size
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.titlesize": 10,
    "pdf.fonttype": 42,   
    "ps.fonttype": 42,
})

import logging
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)

column_width = 3.25

In [ ]:
def imgid_plusxframes(image_id, x):
    record_id = image_id.split('_')[0]
    digits = image_id.split('_')[1]
    updated_digits = int(digits) + x
    updated_image_id = f"{record_id}_{updated_digits:06d}"
    return updated_image_id

# 3x4 Plots

In [ ]:
def visualize_image_ids_collage3(image_ids, data_base_dir=data_base_dir, save_path=None,):
    """
    Visualize a collage of images: 2 rows, each with 2 groups of 3 images.
    The gap between groups and rows is equal-sized visually.
    """
    images_per_group = 3
    num_groups = len(image_ids)
    total_rows = 3
    total_cols = 7  # 3 images + gap + 3 images = 7 columns (gap = 1 column slot)

    # Define relative column and row sizes so gap == same visually
    # 3 columns of images (weight=1 each), 1 small gap (weight=0.15), then 3 more images
    col_widths = [2, 2, 2, 0.1, 2, 2, 2]
    row_heights = [2, 0.1, 2]  # middle row is gap, same ratio as col gap

    # ---- Figure sizing ----
    cm_to_inch = 1 / 2.54
    fig_width_cm = 17.46 #8.25
    fig_width_in = fig_width_cm * cm_to_inch

    # Compute proportional height (based on aspect ratio of grid)
    width_ratio_sum = sum(col_widths)
    height_ratio_sum = sum(row_heights)
    fig_height_in = fig_width_in * (height_ratio_sum / width_ratio_sum)
    #print(fig_width_in, fig_height_in)

    fig = plt.figure(figsize=(fig_width_in, fig_height_in),dpi=300)
    print(fig_width_in, fig_height_in)
    gs = gridspec.GridSpec(
        nrows=3, ncols=7, 
        width_ratios=col_widths, height_ratios=row_heights,
        figure=fig, 
        wspace=0.01,  # eliminate column gaps
        hspace=0.01   # eliminate row gaps
    )

    # Flatten image_ids into groups of 3
    all_handles = []
    all_labels = []
    group_idx = 0
    for row in [0, 2]:  # two image rows (0 and 2, with row 1 as gap)
        for col_group, start_col in enumerate([0, 4]):  # 0 and 4, with col 3 as gap
            if group_idx >= num_groups:
                break

            image_id = image_ids[group_idx]
            for i in range(images_per_group):
                img_arr, legend = imgid_to_imgarray(image_id, annotated=True, actions=False, only_person=True)
                ax = fig.add_subplot(gs[row, start_col + i])
                ax.imshow(img_arr)
                ax.axis("off")
                # Collect legend entries
                if legend:
                    handles, labels = zip(*[(h, h.get_label()) for h in legend])
                    for h, l in zip(handles, labels):
                        if l not in all_labels:  # avoid duplicates
                            all_handles.append(h)
                            all_labels.append(l)

                image_id = imgid_plusxframes(image_id, 100)

            group_idx += 1

    plt.subplots_adjust(
        left=0, right=1, top=1, bottom=0,
        wspace=0, hspace=0
    )
    # ---- Add common legend ----
    desired_order = ['seated', 'seated_ground', 'standing', 'lying', 'bag', 'clothing']  # preferred order for legend
    # Sort according to the desired order
    ordered_pairs = sorted(
        zip(all_labels, all_handles),
        key=lambda x: desired_order.index(x[0]) if x[0] in desired_order else len(desired_order)
    )
    all_labels, all_handles = zip(*ordered_pairs)
    
    # --- Make legend lines thicker ---
    for h in all_handles:
        try:
            h.set_linewidth(2.0)  # adjust thickness here
        except AttributeError:
            pass  # skip if handle isn't a Line2D (e.g., patch, rectangle, etc.)

    if len(all_handles) == 6:
        fig.legend(
            handles=all_handles,
            labels=all_labels,
            loc="lower center",
            bbox_to_anchor=(0.5,-0.11),
            ncol=6,
            #fontsize=12.5,
            frameon=False,)
    
    else:
        fig.legend(
            handles=all_handles,
            labels=all_labels,
            loc="lower center",
            bbox_to_anchor=(0.5,-0.11),
            ncol=len(all_handles),
            #fontsize=12.5,
            frameon=False,)
        

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300, pad_inches=0)

    plt.show()


In [ ]:
image_ids = ['rec1_002612', 'rec25_003735', 'rec10_001000', 'rec20_001059',]
visualize_image_ids_collage3(image_ids)#, save_path='./plots/train_sample.png') 

In [ ]:
image_ids = ['rec29_000869', 'rec28_002014', 'rec26_002007', 'rec30_001580']
visualize_image_ids_collage3(image_ids, save_path='./plots/val_sample.png') 

# 4x1x3

In [ ]:
def visualize_image_ids_collage_onecolumn_recnames(image_ids, data_base_dir=data_base_dir, save_path=None, plot_legend=True):
    """
    Visualize a collage of images: 4 rows, each with 1 group of 3 images.
    The gap between groups and rows is equal-sized visually.
    """
    images_per_group = 3
    num_groups = len(image_ids)
    total_rows = 2* num_groups  #1 image, 1 gap per group
    total_cols = images_per_group

    # Define relative column and row sizes so gap == same visually
    # 3 columns of images (weight=1 each), 1 small gap (weight=0.15), then 3 more images
    col_widths = [2, 2, 2]
#    row_heights = [2, 0.1]*(len(image_ids)-1)+[2]  # middle row is gap, same ratio as col gap
    gap_height = 0.05
    row_heights = [gap_height, 2]* num_groups

    print(row_heights)
    
    # ---- Figure sizing ----
    cm_to_inch = 1 / 2.54
    fig_width_cm = 8.25
    fig_width_in = fig_width_cm * cm_to_inch

    # Compute proportional height (based on aspect ratio of grid)
    width_ratio_sum = sum(col_widths)
    height_ratio_sum = sum(row_heights)
    fig_height_in = fig_width_in * (height_ratio_sum / width_ratio_sum)
    #print(fig_width_in, fig_height_in)

    fig = plt.figure(figsize=(fig_width_in, fig_height_in),dpi=300)
    print(fig_width_in, fig_height_in)
    gs = gridspec.GridSpec(
        nrows=total_rows, ncols=total_cols, 
        width_ratios=col_widths, height_ratios=row_heights,
        figure=fig, 
        wspace=0.01,  # eliminate column gaps
        hspace=0.01   # eliminate row gaps
    )

    # Flatten image_ids into groups of 3
    # de-duplicated legend entries
    label_to_handle = {}

    group_idx = 0
    for r in range(num_groups):
        gap_row = 2 * r
        img_row = 2 * r + 1

        image_id = image_ids[group_idx]
        rec_id = image_id.split('_')[0]
    
        for c in range(images_per_group):
            img_arr, legend = imgid_to_imgarray(image_id, annotated=True)
            print(legend)
            ax = fig.add_subplot(gs[img_row, c])
            ax.imshow(img_arr)
            ax.axis("off")
            if c == 0:
                ax.text(20, 170, rec_id, color='white',fontsize=8)   
            # Collect legend entries
            if legend:
                for handle in legend:
                    try:
                        label = handle.get_label()
                    except Exception:
                        continue
                    if label and label not in label_to_handle:
                        if label == 'seated_ground':
                            label = 'seated (ground)'
                        label_to_handle[label] = handle
                        print(label)

            image_id = imgid_plusxframes(image_id, 100)
        group_idx += 1

    plt.subplots_adjust(
        left=0, right=1, top=1, bottom=0,
        wspace=0, hspace=0
    )
    # ---- Add common legend ----
    if plot_legend:
        desired_order = ['seated', 'seated (ground)', 'standing', 'lying', 'bag', 'clothing']  # preferred order for legend

        if label_to_handle:
            labels = list(label_to_handle.keys())
            handles = [label_to_handle[l] for l in labels]

            ordered_pairs = sorted(zip(labels, handles),
                key=lambda x: desired_order.index(x[0]) if x[0] in desired_order else len(desired_order)
            )
        all_labels, all_handles = map(list, zip(*ordered_pairs))
        
        # --- Make legend lines thicker ---
        for h in all_handles:
            try:
                h.set_linewidth(2.0)  # adjust thickness here
            except AttributeError:
                pass  # skip if handle isn't a Line2D (e.g., patch, rectangle, etc.)

        print(len(all_handles))
        bbox_to_anchor_y = -0.1 if len(all_handles)==3 else -0.15 #-0.09
        fig.legend(
            handles=all_handles,
            labels=all_labels,
            loc="lower center",
            bbox_to_anchor=(0.5,bbox_to_anchor_y),
            ncol=3,
            fontsize=8,
            frameon=False,)

    else:
        print("No legend entries found.")
    
#    fig.suptitle('PMOF: Passenger Monitoring Using Overhead Fisheye Cameras', y=1.03, fontsize=8)

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300, pad_inches=0.015)

    plt.show()

In [ ]:
visualize_image_ids_collage_onecolumn_recnames(image_ids = ['rec5_001409', 'rec28_002012', 'rec30_001580'])#,save_path='./plots/pmof_overview.svg')

# AVSS Poster
Samples:  
'rec1_002612', 'rec5_001409', 'rec10_001000'
'rec12_001793',  'rec20_001059', 'rec22_000948', 
'rec25_003735', 'rec26_002007', 'rec29_000869',
'rec28_002012', 'rec29_000983', 'rec30_001580'




In [ ]:
visualize_image_ids_collage_onecolumn_recnames(image_ids = ['rec1_002612', 'rec5_001409', 'rec6_001990'], plot_legend=False,save_path='./plots/avss_poster_1.svg')

In [ ]:
visualize_image_ids_collage_onecolumn_recnames(image_ids = ['rec7_001608', 'rec10_001000', 'rec12_001793'], plot_legend=False, save_path='./plots/avss_poster_2.svg')


In [ ]:
visualize_image_ids_collage_onecolumn_recnames(image_ids = ['rec15_000568',  'rec20_001059', 'rec22_000948'], plot_legend=False, save_path='./plots/avss_poster_3.svg')


In [ ]:
visualize_image_ids_collage_onecolumn_recnames(image_ids = ['rec25_003735', 'rec26_002007', 'rec27_001988'], plot_legend=False, save_path='./plots/avss_poster_4.svg')


In [ ]:
visualize_image_ids_collage_onecolumn_recnames(image_ids = ['rec28_002012', 'rec29_000983', 'rec30_001580'], plot_legend=True, save_path='./plots/avss_poster_5.svg')
